# Homework 2: Multiple Linear Regression

**Question:** How is salary associated with training hours, after accounting
for experience and education? Which workers earn less than the model predicts
from these characteristics?

Use the salary data from Homework 1 to extend the analysis:

- Inspect types, duplicates, missing values, and variable ranges.
- Define a common estimation sample and construct log salary.
- Explore relationships and investigate unusual observations.
- Add controls, compare coefficients and fit, and discuss uncertainty.
- Predict log salary, distinguish two kinds of interval, and inspect residuals.

Read `README.md` for the task, deadline, and submission instructions.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid")

## 1. Inspect the data and prepare the sample

Each row is a synthetic worker. `salary_k_eur` measures salary in thousands
of euros, `training_hours` measures hours, and experience and education are
measured in years. The other columns are not predictors in this analysis.

The file has no worker ID. Preserve its original DataFrame row labels so
that you can trace a record through the tables and plots. A row label
identifies a record within this file, not a person outside it.

In [ ]:
workers = pd.read_csv("data/career_outcomes.csv")
workers.info()

In [ ]:
print("Rows:", len(workers))
print("Exact duplicate rows:", workers.duplicated().sum())
workers.isna().sum().rename("missing")

In [ ]:
required = ["salary_k_eur", "training_hours", "experience_years", "education_years"]
workers[required].describe().round(2)

Inspect the counts and ranges before making exclusions. Missing values are
not zeros. Repeated values of a predictor do not make two workers duplicates.
A suspicious value needs investigation before it is corrected or deleted.

Compare the available samples, then require complete observations only for
variables used in the candidate models. Use that same sample throughout.

In [ ]:
pd.Series({
    "Salary and training observed": len(
        workers.dropna(subset=["salary_k_eur", "training_hours"])
    ),
    "All model variables observed": len(workers.dropna(subset=required)),
}, name="available_rows")

In [ ]:
complete_sample = workers.dropna(subset=required).copy()
print("Excluded for missing model variables:", len(workers) - len(complete_sample))
print("Rows in the estimation sample:", len(complete_sample))

# Log salary requires a positive salary; check the other ranges too.
assert np.isfinite(complete_sample[required]).all().all(), "Check non-finite values."
assert complete_sample["salary_k_eur"].gt(0).all(), "Salary must be positive."
assert complete_sample[required[1:]].ge(0).all().all(), "Check negative hours or years."
complete_sample["log_salary"] = np.log(complete_sample["salary_k_eur"])

## 2. Explore relationships before estimating

Follow up on the minimum salary from the summary table. Keep its original
row label, inspect its characteristics, and locate it in the scatterplot.

In [ ]:
lowest_salary_row = complete_sample["salary_k_eur"].idxmin()
complete_sample.loc[[lowest_salary_row], required + ["log_salary"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=complete_sample, x="training_hours", y="log_salary",
                alpha=0.65, ax=axes[0])
# Use the original row label to locate the record inspected above.
point = complete_sample.loc[lowest_salary_row]
axes[0].scatter(point["training_hours"], point["log_salary"],
                facecolors="none", edgecolors="red", s=150)
axes[0].annotate(f"Row {lowest_salary_row}",
                 (point["training_hours"], point["log_salary"]),
                 xytext=(8, 8), textcoords="offset points")
sns.boxplot(data=complete_sample, x="education_years", y="log_salary", ax=axes[1])
axes[0].set(title="Training and log salary", xlabel="Training hours", ylabel="Log salary")
axes[1].set(title="Log salary by education", xlabel="Education (years)", ylabel="Log salary")
plt.tight_layout()
plt.show()

**Your interpretation:** Report any exclusions and explain them, including
when no rows were excluded. Describe the plotted relationships and identify
an observation worth investigating. Does being unusual establish a data error?
What would you need to check? Refer to a row label when discussing a record.

Retain unusual observations for the main analysis: this synthetic file gives
us no source evidence for a correction or an outlier-based exclusion.

## 3. Estimate and compare the specifications

Start with training hours, then add experience, then education. The sequence
asks how the training–salary association changes as these characteristics
are held constant. Experience and education enter numerically, with a constant
change in predicted log salary for each additional year.

All models include an intercept and use the common sample. Use HC0
heteroskedasticity-robust standard errors, as in class. `missing="raise"`
stops estimation if an unexpected missing value appears.

In [ ]:
model_1 = smf.ols(
    "log_salary ~ training_hours", data=complete_sample, missing="raise"
).fit(cov_type="HC0")
model_2 = smf.ols(
    "log_salary ~ training_hours + experience_years",
    data=complete_sample, missing="raise"
).fit(cov_type="HC0")
model_3 = smf.ols(
    "log_salary ~ training_hours + education_years",
    data=complete_sample, missing="raise"
).fit(cov_type="HC0")
assert model_1.nobs == model_2.nobs == model_3.nobs == len(complete_sample)

In [ ]:
terms = ["training_hours", "experience_years", "education_years"]
regression_table = pd.DataFrame({
    "Training": model_1.params,
    "+ experience": model_2.params,
    "+ education": model_3.params,
}).reindex(terms)
regression_table.loc["R-squared"] = [model_1.rsquared, model_2.rsquared, model_3.rsquared]
regression_table.loc["Adjusted R-squared"] = [
    model_1.rsquared_adj, model_2.rsquared_adj, model_3.rsquared_adj,
]
regression_table.loc["Observations"] = [model_1.nobs, model_2.nobs, model_3.nobs]
regression_table.round(4).fillna("—")

In [ ]:
estimates = pd.DataFrame({"estimate": model_3.params, "robust_se": model_3.bse})
estimates.drop(index="Intercept").round(4)

**Your interpretation:** Follow the training coefficient across the columns.
State its units and what is held constant in each specification. Explain
the pattern in R-squared and adjusted R-squared, referring to the sample sizes.
What do the robust standard errors tell you? Does the model establish that
additional training causes higher pay?

These are descriptions of fit to the estimation sample. The intercept is
included in estimation but omitted from the comparison table.

## 4. Predict log salary and express uncertainty

Use Model 3 for a worker with 40 training hours, 10 years of experience,
and 16 years of education. Keep predictions and intervals on the log scale.

In [ ]:
proposed_worker = pd.DataFrame({
    "training_hours": [40.0], "experience_years": [10.0], "education_years": [16.0],
})
prediction_log = model_3.get_prediction(proposed_worker).summary_frame(alpha=0.05)
prediction_log.round(4)

**Your interpretation:** Report the predicted log salary and both 95%
intervals. Which interval concerns the mean log salary for workers with
these characteristics, and which concerns one new worker? Explain why
the second is wider.

As in class, the individual prediction interval adds a common estimated
residual variance. HC0 adjusts coefficient uncertainty; it does not make
that residual variance depend on the worker's characteristics.

## 5. Investigate residuals

Define the residual as observed log salary minus fitted log salary.
Inspect the workers with the most negative residuals, then compare
observed and fitted values for the full sample.

In [ ]:
complete_sample["predicted_log_salary"] = model_3.fittedvalues
complete_sample["residual_log_salary"] = model_3.resid
complete_sample.nsmallest(5, "residual_log_salary")[[
    "log_salary", "predicted_log_salary", "residual_log_salary",
    "training_hours", "experience_years", "education_years",
]].round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=complete_sample, x="predicted_log_salary", y="log_salary",
                alpha=0.65, ax=ax)
limits = [
    complete_sample[["log_salary", "predicted_log_salary"]].min().min(),
    complete_sample[["log_salary", "predicted_log_salary"]].max().max(),
]
ax.plot(limits, limits, color="black", linestyle="--")
ax.set(title="Observed versus predicted log salaries",
       xlabel="Predicted log salary", ylabel="Observed log salary")
plt.tight_layout()
plt.show()

**Your interpretation:** Choose one record from the residual table and
explain its residual, giving its row label and using log-salary units.
Does the record investigated before estimation also appear here? Use its
original row label to check. Why does a negative residual not by itself establish that a worker is
unfairly paid? Name a relevant characteristic the model does not include.

## Your review

Briefly document what you changed, why, and how you checked the resulting
analysis. Complete the interpretation prompts with your own explanations
and numerical evidence from the corrected notebook. If you used an LLM,
explain how you checked its suggestions. Restart the kernel, run all cells,
and save the outputs before submitting as described in `README.md`.